# 08 · Equivalence (TOST) + E-value for the main effects

1. **TOST** for the mean Δpain comparison (notebook 01, Finding 2). Equivalence margin = ±0.5 NP1PAIN point — i.e., a 0.5-point or larger group-mean difference would be considered clinically relevant, smaller is noise.
2. **E-value** for the Δslope contrast (notebook 01b, Finding 1). How much unmeasured confounding (in RR-equivalent terms) would be needed to explain away the pre-vs-post slope flip?
3. **E-value** for the responder/worsener risk ratio (DBS vs Never-DBS).

In [1]:
source("helpers/pain_helpers.R")
suppressPackageStartupMessages({ library(dplyr); library(TOSTER); library(EValue) })

delta <- readRDS(file.path(OUT_OBJ, "pain_delta_responder.rds"))
cat("Mediation-cohort n:", nrow(delta), "\n")

Mediation-cohort n: 74 


In [2]:
# TOST on Δpain with margin ±0.5
d_dbs <- delta %>% dplyr::filter(will_receive_dbs) %>% dplyr::pull(delta)
d_ctl <- delta %>% dplyr::filter(!will_receive_dbs) %>% dplyr::pull(delta)
cat("n(DBS):", length(d_dbs), "  n(Never-DBS):", length(d_ctl), "\n")

tost <- TOSTER::tsum_TOST(
  m1 = mean(d_dbs), sd1 = stats::sd(d_dbs), n1 = length(d_dbs),
  m2 = mean(d_ctl), sd2 = stats::sd(d_ctl), n2 = length(d_ctl),
  low_eqbound = -0.5, high_eqbound = 0.5,
  eqbound_type = "raw"
)
print(tost)
save_object(tost, "tost_delta_pain")

n(DBS): 40   n(Never-DBS): 34 



Welch Two Sample t-test

The equivalence test was non-significant, t(64.38) = -0.361, p = 6.4e-01
The null hypothesis test was significant, t(64.38) = -2.451, p = 1.7e-02
NHST: reject null significance hypothesis that the effect is equal to zero 
TOST: don't reject null equivalence hypothesis

TOST Results 
                 t    df p.value
t-test     -2.4513 64.38   0.017
TOST Lower -0.3607 64.38    0.64
TOST Upper -4.5419 64.38 < 0.001

Effect Sizes 
               Estimate     SE               C.I. Conf. Level
Raw             -0.5863 0.2392 [-0.9854, -0.1871]         0.9
Hedges's g(av)  -0.5696 0.2433 [-0.9581, -0.1769]         0.9
Note: SMD confidence intervals are an approximation. See vignette("SMD_calcs").


In [3]:
# E-value for Δslope contrast (Pre-DBS − Post-DBS) per year
# Reading from notebook 01b year-scale: Δ = −0.128, 95% CI (approx) [−0.205, −0.051]
# Treat as a standardised mean difference via LR formula.
# For a continuous outcome, EValue::evalues.OLS() expects effect in sd units.
sl <- readr::read_csv(file.path(OUT_TAB, "lmm_slopes_per_year.csv"), show_col_types = FALSE)
ct <- readr::read_csv(file.path(OUT_TAB, "lmm_slope_contrasts_per_year.csv"), show_col_types = FALSE)
pre_post <- ct %>% dplyr::filter(grepl("Pre-DBS", contrast), grepl("Post-DBS", contrast))
print(pre_post)

# Residual SD proxy: SD of per-year slopes across groups as a rough yardstick
# (E-value for OLS needs sd; we use the table's SE * sqrt(df) approximation)
est <- pre_post$estimate[1]
se  <- pre_post$SE[1]
lo  <- est - 1.96 * se; hi <- est + 1.96 * se
cat(sprintf("Pre−Post slope/yr: est=%.4f  95%% CI [%.4f, %.4f]\n", est, lo, hi))

# NP1PAIN SD from the matched cohort
df_long <- readRDS(file.path(OUT_OBJ, "pain_long.rds"))
sd_pain <- stats::sd(df_long$NP1PAIN, na.rm = TRUE)
cat("SD(NP1PAIN) =", round(sd_pain, 3), "\n")

ev_slope <- EValue::evalues.OLS(est = est, se = se, sd = sd_pain)
print(ev_slope)
save_table(as.data.frame(ev_slope), "evalue_slope_contrast")

# A tibble: 1 × 6
  contrast               estimate     SE    df z.ratio p.value
  <chr>                     <dbl>  <dbl> <dbl>   <dbl>   <dbl>
1 (Pre-DBS) - (Post-DBS)   -0.128 0.0392   Inf   -3.26 0.00315


Pre−Post slope/yr: est=-0.1280  95% CI [-0.2049, -0.0511]


SD(NP1PAIN) = 1.091 


             point     lower     upper
RR       0.8986943 0.8429714 0.9581006
E-values 1.4668900        NA 1.2573769


In [4]:
# E-value for the worsener risk ratio (DBS vs Never-DBS)
worsen_tbl <- table(
  ifelse(delta$will_receive_dbs, "DBS", "Never-DBS"),
  ifelse(delta$delta >= 1, "Worsener", "Not worsener")
)
print(worsen_tbl)
p_dbs <- worsen_tbl["DBS","Worsener"]      / sum(worsen_tbl["DBS",])
p_ctl <- worsen_tbl["Never-DBS","Worsener"] / sum(worsen_tbl["Never-DBS",])
rr <- p_dbs / p_ctl
cat(sprintf("P(worsen | DBS) = %.3f\nP(worsen | Never-DBS) = %.3f\nRR = %.3f\n",
            p_dbs, p_ctl, rr))

# Delta-method CI on log-RR
a <- worsen_tbl["DBS","Worsener"];      b <- sum(worsen_tbl["DBS",]) - a
c <- worsen_tbl["Never-DBS","Worsener"]; d <- sum(worsen_tbl["Never-DBS",]) - c
se_logrr <- sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
lo <- exp(log(rr) - 1.96 * se_logrr); hi <- exp(log(rr) + 1.96 * se_logrr)
cat(sprintf("95%% CI on RR: [%.3f, %.3f]\n", lo, hi))

ev_rr <- EValue::evalues.RR(est = rr, lo = lo, hi = hi)
print(ev_rr)
save_table(as.data.frame(ev_rr), "evalue_worsener_rr")

           
            Not worsener Worsener
  DBS                 35        5
  Never-DBS           22       12


P(worsen | DBS) = 0.125
P(worsen | Never-DBS) = 0.353
RR = 0.354


95% CI on RR: [0.139, 0.905]


             point     lower     upper
RR       0.3541667 0.1386553 0.9046465
E-values 5.0926237        NA 1.4467458
